# Lab 3.4, Build 2: Install the guardrails

Tina's pipeline has three hook points and all three are empty. You fill them, then
run fifteen dev queries in three classes and watch which hook fires.

Run the harness cell first. It retrieves all fifteen queries and prints the top
relevance score per class, which is the measurement your confidence threshold comes
from. Then complete the cell marked **YOUR WORK**.

In [ ]:
# Harness. Nothing here is graded. The grader retrieves, packs, and generates the
# same way, so what you see here is what it sees.
import json
import os
import pathlib
import re
import sys

sys.path.insert(0, "/opt/ara/lib")

from elasticsearch import Elasticsearch
from ara_attrib import split_claims, support

ES = Elasticsearch(os.environ["ES_URL"], api_key=os.environ["ES_API_KEY"],
                   request_timeout=120)
TRACES = pathlib.Path("/home/elastic/.traces")
TRACES.mkdir(exist_ok=True)

COMPLETION_ID = os.environ.get("ARA_INFERENCE_COMPLETION_ID", "cortex-generation")
RERANK_ID = os.environ.get("ARA_RERANK_ID", "")

ANSWER_PROMPT = (
    "You are a compliance assistant for Cortex Bank and Trust. Answer the question "
    "using only the case material and policy text below. Quote every figure exactly "
    "as it appears in the material. If the material does not contain the answer, "
    "reply with the single sentence: I cannot find this information in the case files "
    "or the policy library.\n\n"
    "Material:\n{context}\n\nQuestion: {question}\nAnswer:"
)


def retrieve(query_text, cases=2, policies=1):
    """Case memos and policy chunks for one question, ordered by rerank relevance."""
    passages = []
    plans = (
        ("cortex-cases", "case", {
            "retriever": {"rrf": {"retrievers": [
                {"standard": {"query": {"match": {"body_text": query_text}}}},
                {"standard": {"query": {"semantic": {"field": "body",
                                                     "query": query_text}}}},
            ], "rank_window_size": 50, "rank_constant": 60}},
            "size": cases,
        }),
        ("cortex-policies", "policy", {
            "retriever": {"standard": {"query": {"semantic": {"field": "body",
                                                              "query": query_text}}}},
            "size": policies,
        }),
    )
    for index, source_type, body in plans:
        response = ES.search(index=index, body=body)
        for hit in response["hits"]["hits"]:
            source = hit.get("_source", {})
            text = source.get("body_text") or source.get("body") or ""
            if isinstance(text, dict):
                text = text.get("text", "")
            passages.append({
                "passage_id": str(source.get("case_id") or source.get("doc_id")
                                  or hit["_id"]),
                "source_type": source_type,
                "text": text,
                "score": float(hit.get("_score", 0.0) or 0.0),
            })
    if RERANK_ID and passages:
        response = ES.inference.inference(inference_id=RERANK_ID, body={
            "query": query_text,
            "input": [(p["text"] or "")[:2000] for p in passages],
        })
        body = response.body if hasattr(response, "body") else response
        for entry in body.get("rerank") or []:
            position = int(entry.get("index", -1))
            if 0 <= position < len(passages):
                passages[position]["score"] = float(entry.get("relevance_score", 0.0) or 0.0)
        passages.sort(key=lambda p: p["score"], reverse=True)
    return passages


def pack(passages, per_passage_chars=1400):
    """The context string the answer prompt carries."""
    return "\n\n".join(
        f"[{p['passage_id']} | {p['source_type']}]\n{(p['text'] or '')[:per_passage_chars]}"
        for p in passages
    )


def ask(question, passages):
    """Tina answers at temperature 0 over exactly these passages."""
    response = ES.inference.inference(inference_id=COMPLETION_ID, body={
        "input": ANSWER_PROMPT.format(context=pack(passages), question=question),
        "task_settings": {"temperature": 0},
    })
    body = response.body if hasattr(response, "body") else response
    return (body.get("completion") or [{}])[0].get("result", "") or ""


DEV = json.loads(
    pathlib.Path("/home/elastic/dev-sets/dev-guardrail-queries.json").read_text()
)

DOLLAR_RE = re.compile(r"\$\s?\d[\d,]*(?:\.\d+)?")
PERCENT_RE = re.compile(r"\b\d+(?:\.\d+)?\s?(?:%|per\s?cent|percent)\b", re.IGNORECASE)
DAYS_RE = re.compile(r"\b\d+(?:\.\d+)?[-\s]+(?:calendar\s+|business\s+)?(?:day|days)\b",
                     re.IGNORECASE)
NUM_RE = re.compile(r"\$?\d[\d,]*(?:\.\d+)?%?")
STOP = {"the", "a", "an", "of", "and", "or", "to", "in", "on", "for", "was", "were",
        "is", "are", "be", "been", "that", "this", "with", "by", "at", "from", "as",
        "it", "its", "must", "may", "not", "no", "any", "all", "which", "within",
        "during", "over", "than", "then", "there", "their", "has", "had", "have",
        "after", "before", "one"}


def figures_in(text):
    """Dollar figures, day counts, and percentages. The grader uses this function."""
    found = []
    for pattern in (DOLLAR_RE, PERCENT_RE, DAYS_RE):
        found.extend(match.group(0).strip() for match in pattern.finditer(text or ""))
    return found


def _content_words(text):
    words = re.findall(r"[A-Za-z][A-Za-z'-]+", (text or "").lower())
    return {w for w in words if w not in STOP and len(w) > 2}


def reference_attribution(claims, passages, overlap_target=0.40, figure_target=0.20):
    """The attribution validate_output is handed. Deterministic, no model call.

    A claim is attributed to a passage when every number in the claim appears in that
    passage and enough of the claim's content words do too. This is the attributor
    the grader uses here, so Build 2 grades your hooks rather than your Build 1 code.
    """
    out = []
    for claim in claims:
        numbers = {n.replace(",", "").replace(" ", "") for n in NUM_RE.findall(claim or "")}
        words = _content_words(claim)
        target = figure_target if numbers else overlap_target
        best_id, best_score = "UNSUPPORTED", 0.0
        for passage in passages:
            text = passage.get("text") or ""
            if numbers and not all(n in text.replace(",", "") for n in numbers):
                continue
            if not words:
                continue
            score = len(words & _content_words(text)) / len(words)
            if score > best_score:
                best_id, best_score = passage.get("passage_id", ""), score
        out.append({"claim": claim,
                    "passage_id": best_id if best_score >= target else "UNSUPPORTED"})
    return out


# Score survey: the top relevance score per query, grouped by class. This is the
# measurement your confidence threshold comes from.
SURVEY = {}
for query in DEV:
    passages = retrieve(query["query_text"])
    SURVEY[query["query_id"]] = {
        "query_class": query["query_class"],
        "passages": passages,
        "top": passages[0]["score"] if passages else 0.0,
        "second": passages[1]["score"] if len(passages) > 1 else 0.0,
    }

for query_class in ("answerable", "unanswerable", "out_of_scope"):
    rows = [(qid, row) for qid, row in SURVEY.items()
            if row["query_class"] == query_class]
    tops = [row["top"] for _qid, row in rows]
    print(f"\n{query_class}  ({len(rows)} queries)")
    for qid, row in rows:
        print(f"  {qid}  top {row['top']:.3f}  margin {row['top'] - row['second']:.3f}")
    if tops:
        print(f"  range {min(tops):.3f} to {max(tops):.3f}")

## Your three hooks

`scope_check` runs before retrieval, `confidence_fallback` after it, and
`validate_output` after generation. Each one returns what the pipeline delivers when
it fires.

In [ ]:
# YOUR WORK: the three guardrail hooks.
CONFIDENCE_THRESHOLD = 0.0  # TODO: the value your score survey supports

REFUSAL = "I cannot find this information in the case files or the policy library."


def scope_check(query):
    """Return (in_scope, decline_message). Runs before anything is retrieved."""
    # TODO: decide from the query text alone. A term list costs nothing; the
    # completion endpoint at temperature 0 is available if you want a classifier.
    # Test against the answerable queries too: declining a real question is worse
    # than having no scope hook at all.
    return (True, "")


def confidence_fallback(results, threshold=CONFIDENCE_THRESHOLD):
    """Return (should_fallback, fallback_message). Runs after retrieval.

    results entries carry passage_id, source_type, text, and score, highest first.
    """
    # TODO: two signals are available, the top score and the margin over the second
    # result. An empty result set is its own case. Setting the bar too high costs you
    # answerable questions, and the grader measures both directions.
    return (False, "")


def validate_output(answer, attribution):
    """Return the text Tina is allowed to deliver.

    attribution is one entry per claim, in claim order:
      [{"claim": "...", "passage_id": "case-structuring-001"},
       {"claim": "...", "passage_id": "UNSUPPORTED"}]
    """
    # TODO: pass the answer through when every claim is attributed. Remove the
    # unattributed claims and append a short note when some are not. Refuse when
    # nothing worth delivering is left, and decide what "nothing" means before you
    # write that branch.
    return answer

## Run the three classes

The pipeline stops at the first hook that fires, so the hook column tells you where
each query ended.

In [ ]:
# Run the fifteen dev queries through the pipeline with your hooks in place.
ROWS = []

for query in DEV:
    text = query["query_text"]
    row = {"query_id": query["query_id"], "query_class": query["query_class"],
           "hook": "", "top_score": 0.0, "figures": [], "delivered": ""}

    in_scope, decline = scope_check(text)
    if not in_scope:
        row.update(hook="scope_check", delivered=decline, figures=figures_in(decline))
        ROWS.append(row)
        continue

    passages = SURVEY[query["query_id"]]["passages"]
    row["top_score"] = round(passages[0]["score"], 4) if passages else 0.0

    fires, message = confidence_fallback(passages, CONFIDENCE_THRESHOLD)
    if fires:
        row.update(hook="confidence_fallback", delivered=message,
                   figures=figures_in(message))
        ROWS.append(row)
        continue

    answer = ask(text, passages)
    claims = split_claims(answer)
    attribution = reference_attribution(claims, passages)
    delivered = validate_output(answer, attribution)
    row.update(hook="validate_output", delivered=delivered,
               figures=figures_in(delivered),
               unattributed=sum(1 for a in attribution
                                if a["passage_id"] == "UNSUPPORTED"))
    ROWS.append(row)

print(f"{len(ROWS)} queries run. Which hook fired:")
for row in ROWS:
    print(f"  {row['query_id']}  {row['query_class']:<13} {row['hook'] or 'none':<20} "
          f"top {row['top_score']:.3f}  figures {len(row['figures'])}")

## Read the classes separately

A total hides which class is failing and why. The Defend asks you which class your
Tina handled worst.

In [ ]:
# Read the three classes on their own terms. They pass for different reasons.
PER_CLASS = {"answerable": 0, "unanswerable": 0, "out_of_scope": 0}
by_id = {q["query_id"]: q for q in DEV}

print("answerable: the exact figure has to survive into the delivered text")
for row in [r for r in ROWS if r["query_class"] == "answerable"]:
    gold = by_id[row["query_id"]].get("gold_literal", "")
    held = gold in row["delivered"]
    clean = row.get("unattributed", 0) == 0
    PER_CLASS["answerable"] += int(held)
    print(f"  {row['query_id']}  gold {'yes' if held else 'NO':<3}  "
          f"unattributed {row.get('unattributed', '-')}  hook {row['hook']}")

print("\nunanswerable: zero figures, and a guardrail has to act")
for row in [r for r in ROWS if r["query_class"] == "unanswerable"]:
    clean = not row["figures"]
    PER_CLASS["unanswerable"] += int(clean)
    print(f"  {row['query_id']}  figures {row['figures'] or 'none'}  hook {row['hook']}")
    if row["figures"]:
        print(f"    delivered: {row['delivered'][:150]}")

print("\nout of scope: declined, and nothing retrieved first")
for row in [r for r in ROWS if r["query_class"] == "out_of_scope"]:
    declined = row["hook"] == "scope_check"
    PER_CLASS["out_of_scope"] += int(declined)
    print(f"  {row['query_id']}  {'declined' if declined else 'ANSWERED'}  "
          f"hook {row['hook']}")

print(f"\nper class: {PER_CLASS}")
print("Five of five answerable, zero figures on the unanswerable class, and five of "
      "five declined is the signal that the held-out run will clear the targets.")

In [ ]:
# Save the results file the grader reads.
payload = {
    "threshold": CONFIDENCE_THRESHOLD,
    "per_class": PER_CLASS,
    "per_query": [{k: v for k, v in row.items() if k != "delivered"} for row in ROWS],
}
(TRACES / "guardrail-results.json").write_text(json.dumps(payload, indent=2))
print(f"guardrail-results.json written with {len(ROWS)} rows.")
print("Select Check.")